# Read percept_single.csv with a CID column, and find IsomericSMILES

In [1]:
import math
import time
from typing import Dict, Any, Optional, Tuple, List

import pandas as pd
import requests

IN_CSV  = "percept_single.csv"
OUT_CSV = "percept_single_with_pubchem.csv"

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
PUGVIEW_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "CID->PubChem/2.0", "Accept": "application/json"})
TIMEOUT = 20
BATCH_SIZE = 100
SLEEP_BETWEEN_BATCHES = 0.05

def _get_json(url: str, retries: int = 2, backoff: float = 0.6) -> Optional[dict]:
    for i in range(retries + 1):
        try:
            r = SESSION.get(url, timeout=TIMEOUT)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (400, 404):
                return None
        except requests.RequestException:
            pass
        time.sleep(backoff * (2 ** i))
    return None

def normalize_cid(value: Any) -> Optional[int]:
    """Return an integer CID or None."""
    if value is None:
        return None
    s = str(value).strip()
    if not s:
        return None
    # Remove spurious .0 from CSV float parsing
    try:
        # try int directly
        return int(s)
    except ValueError:
        # try float that is integral
        try:
            f = float(s)
            if math.isfinite(f) and abs(f - round(f)) < 1e-9 and f >= 0:
                return int(round(f))
        except ValueError:
            return None
    return None

def extract_props(j: Dict[str, Any]) -> Dict[int, Tuple[Optional[str], Optional[str]]]:
    out: Dict[int, Tuple[Optional[str], Optional[str]]] = {}
    try:
        props = j["PropertyTable"]["Properties"]
    except Exception:
        return out
    for rec in props:
        cid = rec.get("CID")
        if cid is None:
            continue
        name = rec.get("IUPACName")
        smi  = rec.get("IsomericSMILES") or rec.get("CanonicalSMILES")
        out[int(cid)] = (name, smi)
    return out

def pugview_title_and_smiles(cid: int) -> Tuple[Optional[str], Optional[str]]:
    j = _get_json(f"{PUGVIEW_BASE}/{cid}/JSON/")
    if not j:
        return None, None

    # Title
    name = None
    try:
        name = j["Record"]["RecordTitle"]
    except Exception:
        name = None

    # SMILES scan
    targets = {"ISOMERIC SMILES", "CANONICAL SMILES", "SMILES"}
    found = {}

    def pull_value(info: dict) -> Optional[str]:
        sv = info.get("StringValue")
        if isinstance(sv, str) and sv:
            return sv.strip()
        val = info.get("Value")
        if isinstance(val, dict):
            swm = val.get("StringWithMarkup")
            if isinstance(swm, list) and swm:
                s = swm[0].get("String")
                if isinstance(s, str) and s:
                    return s.strip()
        s = info.get("String")
        return s.strip() if isinstance(s, str) and s else None

    def walk(node):
        if isinstance(node, dict):
            label = None
            nm = node.get("Name")
            if isinstance(nm, str):
                label = nm.strip().upper()
            th = node.get("TOCHeading")
            if isinstance(th, str):
                label = th.strip().upper() or label
            if label in targets:
                info_list = node.get("Information")
                if isinstance(info_list, list):
                    for info in info_list:
                        if isinstance(info, dict):
                            val = pull_value(info)
                            if val:
                                found.setdefault(label, val)
            for v in node.values():
                walk(v)
        elif isinstance(node, list):
            for it in node:
                walk(it)

    walk(j)

    for key in ("ISOMERIC SMILES", "CANONICAL SMILES", "SMILES"):
        if key in found:
            return name, found[key]
    return name, None

def batched_lookup(cids: List[int]) -> Dict[int, Tuple[Optional[str], Optional[str]]]:
    result: Dict[int, Tuple[Optional[str], Optional[str]]] = {}
    unique_cids = sorted(set([c for c in cids if isinstance(c, int)]))
    total = len(unique_cids)
    if total == 0:
        return result

    # Batched property queries
    for start in range(0, total, BATCH_SIZE):
        batch = unique_cids[start:start+BATCH_SIZE]
        cid_list = ",".join(str(c) for c in batch)
        url = f"{PUBCHEM_BASE}/compound/cid/{cid_list}/property/IUPACName,IsomericSMILES,CanonicalSMILES/JSON"
        j = _get_json(url)
        if j:
            result.update(extract_props(j))
        print(f"[batch {start//BATCH_SIZE + 1}] props fetched for {len(batch)} CIDs", flush=True)
        time.sleep(SLEEP_BETWEEN_BATCHES)

    # Fallback via PUG-View for any CID still missing name or smiles
    missing = [c for c in unique_cids if c not in result or not result[c][0] or not result[c][1]]
    if missing:
        print(f"Falling back to PUG-View for {len(missing)} CIDs...")
    for idx, cid in enumerate(missing, 1):
        old = result.get(cid, (None, None))
        name0, smi0 = old
        name1, smi1 = pugview_title_and_smiles(cid)
        name = name0 or name1 or ""
        smi  = smi0 or smi1 or ""
        result[cid] = (name, smi)
        if idx % 50 == 0 or idx == len(missing):
            print(f"  PUG-View looked up {idx}/{len(missing)}", flush=True)
        time.sleep(0.02)

    return result

def main():
    df = pd.read_csv(IN_CSV)
    if "CID" not in df.columns:
        raise SystemExit("Missing required column: CID")

    # Normalize CIDs
    df["_CID_INT"] = df["CID"].apply(normalize_cid)
    n_before = len(df)
    df_valid = df.dropna(subset=["_CID_INT"]).copy()
    df_valid["_CID_INT"] = df_valid["_CID_INT"].astype(int)
    print(f"CIDs: {len(df_valid)}/{n_before} rows have valid integer CIDs.")

    # Lookup
    mapping = batched_lookup(df_valid["_CID_INT"].tolist())

    # Build output columns
    names = []
    smiles = []
    filled_names = filled_smiles = 0
    for cid_val in df["_CID_INT"]:
        if pd.isna(cid_val):
            names.append("")
            smiles.append("")
            continue
        pair = mapping.get(int(cid_val), ("", ""))
        nm, sm = (pair[0] or ""), (pair[1] or "")
        names.append(nm)
        smiles.append(sm)
        if nm: filled_names += 1
        if sm: filled_smiles += 1

    insert_at = df.columns.get_loc("CID") + 1
    df.insert(insert_at, "PUBCHEM_NAME", names)
    df.insert(insert_at + 1, "PUBCHEM_ISOMERIC_SMILES", smiles)
    df = df.drop(columns=["_CID_INT"])

    df.to_csv(OUT_CSV, index=False)
    print(f"\n Wrote {OUT_CSV}")
    print(f"Filled PUBCHEM_NAME for {filled_names}/{len(df)} rows")
    print(f"Filled PUBCHEM_ISOMERIC_SMILES for {filled_smiles}/{len(df)} rows")

if __name__ == "__main__":
    main()


CIDs: 338/338 rows have valid integer CIDs.
[batch 1] props fetched for 100 CIDs
[batch 2] props fetched for 100 CIDs
[batch 3] props fetched for 100 CIDs
[batch 4] props fetched for 38 CIDs
Falling back to PUG-View for 338 CIDs...
  PUG-View looked up 50/338
  PUG-View looked up 100/338
  PUG-View looked up 150/338
  PUG-View looked up 200/338
  PUG-View looked up 250/338
  PUG-View looked up 300/338
  PUG-View looked up 338/338

 Wrote percept_single_with_pubchem.csv
Filled PUBCHEM_NAME for 338/338 rows
Filled PUBCHEM_ISOMERIC_SMILES for 338/338 rows
